# CSV-Aufbereitung · Werkzeug für die `CONFIG` des ML-Skripts (Variante A)

Eine kleine Gradio-App: CSV per Drag & Drop laden, Spalten analysieren, Unnötiges entfernen, fehlende Zeilen optional bereinigen, Zielspalte benennen und einen **fertig-numerischen** Datensatz nach `Datasets/` speichern.

Kategoriale Feature-Spalten werden direkt **One-Hot-codiert** (`0/1`), damit der Datensatz auch anderweitig direkt weiterverwendbar ist. Jede Umwandlung wird in einer `*_schema.json` neben der CSV dokumentiert (Schema / Data Card).

In [1]:
# %pip install gradio pandas

In [2]:
import os
import json
from datetime import datetime
import pandas as pd
import gradio as gr

DATASETS_DIR = "Datasets"
os.makedirs(DATASETS_DIR, exist_ok=True)

# ---- Reine Analyse-/Aufbereitungs-Logik (ohne UI) -------------------------
def analysiere(df):
    """Übersichtstabelle: Typ, Beispiele, Fehlend, Verschiedene, Hinweis."""
    n = len(df); zeilen = []
    for col in df.columns:
        s = df[col]
        is_num    = pd.api.types.is_numeric_dtype(s)
        n_missing = int(s.isna().sum())
        n_unique  = int(s.nunique(dropna=True))
        beispiele = ", ".join(map(str, s.dropna().unique()[:2]))
        if   n_unique <= 1:    hinweis = "konstant -> Löschkandidat"
        elif is_num:           hinweis = "numerisch"
        elif n_unique == n:    hinweis = "alle Werte verschieden -> evtl. ID/Text"
        else:                  hinweis = "kategorial (Text)"
        zeilen.append([col, str(s.dtype), beispiele, n_missing, n_unique, hinweis])
    return pd.DataFrame(zeilen,
        columns=["Spalte", "Typ", "Beispielwerte", "Fehlend", "Verschiedene", "Hinweis"])

def info_text(df):
    return (f"**{df.shape[0]} Zeilen · {df.shape[1]} Spalten · "
            f"{int(df.isna().sum().sum())} fehlende Werte insgesamt**")

def ist_klassifikation(df, ziel):
    """Text-Ziel = Klassifikation. Zahl-Ziel nur, wenn es wenige Stufen hat."""
    s = df[ziel]
    if not pd.api.types.is_numeric_dtype(s):
        return True
    return s.nunique() <= max(20, int(0.05 * len(s)))

def aufbereiten(df, ziel):
    """Variante A: kategoriale FEATURES -> 0/1-Spalten, Ziel bleibt Label am Ende.
    Liefert den codierten DataFrame, ein Doku-Schema und die Zahl entfernter Zeilen."""
    vorher = len(df)
    df = df.dropna()                                   # garantiert sauberen Datensatz
    entfernt = vorher - len(df)

    feats     = [c for c in df.columns if c != ziel]
    num_feats = [c for c in feats if pd.api.types.is_numeric_dtype(df[c])]
    kat_feats = [c for c in feats if c not in num_feats]

    codiert = pd.get_dummies(df, columns=kat_feats, dtype=int)   # dtype=int -> echte 0/1
    kat_doku = {k: sorted([c for c in codiert.columns if c.startswith(k + "_")])
                for k in kat_feats}
    dummy_cols = [c for k in kat_feats for c in kat_doku[k]]

    spalten = num_feats + dummy_cols + [ziel]          # Ziel ans Ende
    codiert = codiert[spalten]

    schema = {
        "erstellt_am": datetime.now().isoformat(timespec="seconds"),
        "werkzeug": "CSV-Aufbereitung (Variante A)",
        "zielspalte": ziel,
        "ziel_bleibt_label": True,
        "zeilen_mit_fehlern_entfernt": int(entfernt),
        "numerische_merkmale": num_feats,
        "kategoriale_merkmale_codiert": kat_doku,
        "spalten_endgueltig": spalten,
        "zeilen": int(len(codiert)),
    }
    return codiert, schema, entfernt


c:\Users\thoma\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---- Callbacks (verbinden Logik mit der Oberfläche) -----------------------
def laden(file):
    if file is None:
        return None, None, gr.update(choices=[]), gr.update(choices=[]), "", "aufbereitet.csv"
    pfad = file if isinstance(file, str) else file.name      # versionsrobust
    df = pd.read_csv(pfad)
    cols = df.columns.tolist()
    stem = os.path.splitext(os.path.basename(pfad))[0]
    return (df, analysiere(df),
            gr.update(choices=cols, value=[]),
            gr.update(choices=cols, value=cols[-1] if cols else None),
            info_text(df), f"{stem}_aufbereitet.csv")

def entfernen(df, cols):
    if df is None:
        return df, None, gr.update(), gr.update(), ""
    df = df.drop(columns=[c for c in cols if c in df.columns])
    rest = df.columns.tolist()
    return (df, analysiere(df),
            gr.update(choices=rest, value=[]), gr.update(choices=rest), info_text(df))

def bereinigen(df):
    if df is None:
        return df, None, ""
    vorher = len(df); df = df.dropna(); entfernt = vorher - len(df)
    return df, analysiere(df), info_text(df) + f"  ·  **{entfernt} Zeilen** mit fehlenden Werten entfernt."

def speichern(df, ziel, dateiname):
    if df is None:
        return None, "Bitte zuerst eine CSV laden."
    if not ziel or ziel not in df.columns:
        return None, "Bitte eine gültige Zielspalte wählen."
    if not ist_klassifikation(df, ziel):
        return None, (f"Die Spalte **{ziel}** wirkt kontinuierlich (sehr viele verschiedene Zahlenwerte). "
                      "Dieses Werkzeug ist nur für **Klassifikation** gedacht – bitte ein diskretes Ziel wählen.")

    codiert, schema, entfernt = aufbereiten(df, ziel)

    if not dateiname:                  dateiname = "aufbereitet.csv"
    if not dateiname.endswith(".csv"): dateiname += ".csv"
    csv_pfad  = os.path.join(DATASETS_DIR, dateiname)
    json_pfad = csv_pfad[:-4] + "_schema.json"
    schema["ziel_datei"] = csv_pfad

    codiert.to_csv(csv_pfad, index=False)
    with open(json_pfad, "w", encoding="utf-8") as f:
        json.dump(schema, f, ensure_ascii=False, indent=2)

    kat_doku = schema["kategoriale_merkmale_codiert"]
    txt  = (f"**Gespeichert:**  \n"
            f"`{csv_pfad}`  ({codiert.shape[0]} Zeilen, {codiert.shape[1]} Spalten) – "
            f"alle Merkmale numerisch.  \n"
            f"`{json_pfad}`  – Dokumentation der Umwandlung (Schema).  \n\n")
    if entfernt:
        txt += f"{entfernt} Zeilen mit fehlenden Werten wurden beim Speichern entfernt.  \n"
    if kat_doku:
        teile = [f"`{k}` → {', '.join('`'+c+'`' for c in v)}" for k, v in kat_doku.items()]
        txt += "**One-Hot-codiert:**  \n" + "  \n".join(teile) + "  \n"
    else:
        txt += "Keine kategorialen Merkmale – es wurde nichts codiert.  \n"
    txt += f"Zielspalte **{ziel}** steht am Ende und bleibt als Klartext-Label erhalten."
    return codiert.head(10), txt

# ---- Oberfläche ------------------------------------------------------------
with gr.Blocks(title="CSV-Aufbereitung") as app:
    gr.Markdown("# CSV-Aufbereitung für das ML-Skript\n"
                "Lade eine CSV, prüfe die Spalten, entferne Unnötiges, benenne das Ziel "
                "und speichere einen **fertig-numerischen, sauberen** Datensatz nach `Datasets/`.\n\n"
                "*Kategoriale Feature-Spalten werden direkt zu `0/1`-Spalten umgewandelt "
                "(One-Hot). Die Umwandlung wird in einer begleitenden `*_schema.json` dokumentiert. "
                "Die Zielspalte bleibt als Klartext-Label erhalten.*")

    state = gr.State()
    file_in = gr.File(label="CSV per Drag & Drop ablegen", file_types=[".csv"])
    info_md = gr.Markdown()
    analyse_df = gr.Dataframe(label="Spaltenanalyse", interactive=False, wrap=True)

    with gr.Row():
        drop_select = gr.Dropdown(label="Spalten entfernen", multiselect=True, choices=[])
        drop_btn = gr.Button("Ausgewählte Spalten entfernen")
    clean_btn = gr.Button("Zeilen mit fehlenden Werten entfernen")

    with gr.Row():
        target_select = gr.Dropdown(label="Zielspalte (Target)", choices=[])
        name_in = gr.Textbox(label="Dateiname zum Speichern", value="aufbereitet.csv")
    save_btn = gr.Button("Aufbereiten & in Datasets/ speichern", variant="primary")

    status_md = gr.Markdown()
    preview_df = gr.Dataframe(label="Vorschau (numerisch, Target am Ende)", interactive=False, wrap=True)

    file_in.change(laden, inputs=file_in,
                   outputs=[state, analyse_df, drop_select, target_select, info_md, name_in])
    drop_btn.click(entfernen, inputs=[state, drop_select],
                   outputs=[state, analyse_df, drop_select, target_select, info_md])
    clean_btn.click(bereinigen, inputs=state, outputs=[state, analyse_df, info_md])
    save_btn.click(speichern, inputs=[state, target_select, name_in],
                   outputs=[preview_df, status_md])

if __name__ == "__main__":
    app.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\thoma\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\thoma\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\thoma\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
